In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz



In [0]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

## Leitura dos dados na camada silver

In [0]:
path = "hackathon2025.silver.base_atraso"

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

data_exec_inicial = 202410

# converte YYYYMM -> date
data_dt = datetime.strptime(str(data_exec_inicial), "%Y%m")

# subtrai 12 meses
data_exec_final = int((data_dt - relativedelta(months=12)).strftime("%Y%m"))
data_exec_final

In [0]:
from pyspark.sql.functions import col
df_book_atraso_01 = (
    spark.read
         .table(path)
         .filter(col("SAFRA") == data_exec_inicial)
         .select("NUM_CPF")
         .distinct()
)
df_book_atraso_01.createOrReplaceTempView("df_book_atraso_01")

df_book_atraso_01.count()

In [0]:
display(df_book_atraso_01.limit(10))

In [0]:
df_book_atraso_02 = (
    spark.read
         .table(path)
         .filter(
             (col("SAFRA") >= data_exec_final) &
             (col("SAFRA") <= data_exec_inicial)
         )
)
df_book_atraso_02.count()

In [0]:

display(df_book_atraso_02.limit(20))

## Criando flag de janela

In [0]:
df_book_atraso_02.createOrReplaceTempView("df_transacoes")

df_temp_01 = spark.sql("""
WITH base AS (
    SELECT
        *,
        TO_DATE(CONCAT(SAFRA, '01'), 'yyyyMMdd') AS data_dt
    FROM df_transacoes
)

SELECT
    *,
    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -1)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U1M,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -3)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U3M,


    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -6)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U6M,

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -9)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U9M,    

    CASE
        WHEN data_dt BETWEEN
             ADD_MONTHS(MAX(data_dt) OVER (PARTITION BY NUM_CPF), -12)
             AND MAX(data_dt) OVER (PARTITION BY NUM_CPF)
        THEN 1 ELSE 0
    END AS U12M

FROM base
ORDER BY NUM_CPF, SAFRA
""")

df_temp_01.createOrReplaceTempView("df_temp_01")
df_temp_01.count()


In [0]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM df_temp_01
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM df_temp_01 r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [0]:
df_resultado = contagem_percentual("IND_PCCR")
display(df_resultado)

In [0]:
print('lista de colunas para tipar')

for col_name in spark.table("df_temp_01").columns:
    if col_name.startswith("VAL_"):
        print(f"{col_name}")

In [0]:
print('lista de colunas para tipar')

for col_name in spark.table("df_temp_01").columns:
    if col_name.startswith("DW_FAIXA_"):
        print(f"{col_name}")

## Criando variáveis explicativas de primeira camada

In [0]:
df_temp_02 = spark.sql("""

SELECT
    NUM_CPF,
    round(avg(case when U1M = 1 then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_FAT_LIQUIDO_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_FAT_LIQUIDO_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_FAT_LIQUIDO_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_FAT_LIQUIDO_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_FAT_LIQUIDO_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_FAT_CREDITO_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_FAT_CREDITO_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_FAT_CREDITO_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_FAT_CREDITO_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_FAT_CREDITO_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_MULTA_CANCELAMENTO_ATRASO,
    round(avg(case when U3M = 1 then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_MULTA_CANCELAMENTO_ATRASO,
    round(avg(case when U6M = 1 then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_MULTA_CANCELAMENTO_ATRASO,
    round(avg(case when U9M = 1 then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_MULTA_CANCELAMENTO_ATRASO,
    round(avg(case when U12M = 1 then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_MULTA_CANCELAMENTO_ATRASO,
    round(avg(case when U1M = 1 then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_FAT_LIQ_JM_MC_ATRASO,
    
    round(avg(case when U1M = 1 then DW_FAIXA_AGING_FATURA else NULL end),2) as VL_MED_U1M_FX_AGING_FAT_ATRASO,
    round(avg(case when U3M = 1 then DW_FAIXA_AGING_FATURA else NULL end),2) as VL_MED_U3M_FX_AGING_FAT_ATRASO,
    round(avg(case when U6M = 1 then DW_FAIXA_AGING_FATURA else NULL end),2) as VL_MED_U6M_FX_AGING_FAT_ATRASO,
    round(avg(case when U9M = 1 then DW_FAIXA_AGING_FATURA else NULL end),2) as VL_MED_U9M_FX_AGING_FAT_ATRASO,
    round(avg(case when U12M = 1 then DW_FAIXA_AGING_FATURA else NULL end),2) as VL_MED_U12M_FX_AGING_FAT_ATRASO,

    round(avg(case when U1M = 1 then DW_FAIXA_AGING_DIVIDA else NULL end),2) as VL_MED_U1M_FX_AGING_DIVIDA_ATRASO,
    round(avg(case when U3M = 1 then DW_FAIXA_AGING_DIVIDA else NULL end),2) as VL_MED_U3M_FX_AGING_DIVIDA_ATRASO,
    round(avg(case when U6M = 1 then DW_FAIXA_AGING_DIVIDA else NULL end),2) as VL_MED_U6M_FX_AGING_DIVIDA_ATRASO,
    round(avg(case when U9M = 1 then DW_FAIXA_AGING_DIVIDA else NULL end),2) as VL_MED_U9M_FX_AGING_DIVIDA_ATRASO,
    round(avg(case when U12M = 1 then DW_FAIXA_AGING_DIVIDA else NULL end),2) as VL_MED_U12M_FX_AGING_DIVIDA_ATRASO,

    round(avg(case when U1M = 1 then DW_FAIXA_TEMPO_BASE else NULL end),2) as VL_MED_U1M_FX_TEMPO_BASE_ATRASO,
    round(avg(case when U3M = 1 then DW_FAIXA_TEMPO_BASE else NULL end),2) as VL_MED_U3M_FX_TEMPO_BASE_ATRASO,
    round(avg(case when U6M = 1 then DW_FAIXA_TEMPO_BASE else NULL end),2) as VL_MED_U6M_FX_TEMPO_BASE_ATRASO,
    round(avg(case when U9M = 1 then DW_FAIXA_TEMPO_BASE else NULL end),2) as VL_MED_U9M_FX_TEMPO_BASE_ATRASO,
    round(avg(case when U12M = 1 then DW_FAIXA_TEMPO_BASE else NULL end),2) as VL_MED_U12M_FX_TEMPO_BASE_ATRASO,

    round(avg(case when U1M = 1 then DW_FAIXA_AGING_PROX_FECH else NULL end),2) as VL_MED_U1M_FX_AGING_PROX_FECH_ATRASO,
    round(avg(case when U3M = 1 then DW_FAIXA_AGING_PROX_FECH else NULL end),2) as VL_MED_U3M_FX_AGING_PROX_FECH_ATRASO,
    round(avg(case when U6M = 1 then DW_FAIXA_AGING_PROX_FECH else NULL end),2) as VL_MED_U6M_FX_AGING_PROX_FECH_ATRASO,
    round(avg(case when U9M = 1 then DW_FAIXA_AGING_PROX_FECH else NULL end),2) as VL_MED_U9M_FX_AGING_PROX_FECH_ATRASO,
    round(avg(case when U12M = 1 then DW_FAIXA_AGING_PROX_FECH else NULL end),2) as VL_MED_U12M_FX_AGING_PROX_FECH_ATRASO,

    --VARIAVEIS IND_PDD
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,

    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PDD = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_PDD = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_PDD = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_PDD = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_PDD = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,

    --VARIAVEIS IND_WO
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_WO_R_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_WO_R_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_WO_R_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_WO_R_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_WO_R_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'R' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'R' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'R' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'R' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'R' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,


    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_WO_W_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_WO_W_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_WO_W_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_WO_W_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_WO_W_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_WO = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_WO = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_WO = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_WO = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_WO = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,

    --VARIAVEIS IND_PCCR
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'W' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,

    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_PCCR = 'C' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,



    --VARIAVEIS IND_ACA
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,

    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_ACA = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_ACA = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_ACA = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_ACA = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_ACA = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,

    --VARIAVEIS IND_PRIMEIRA_FAT
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,

    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_PRIMEIRA_FAT = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,



    --VARIAVEIS IND_FRAUDE
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'N' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,


    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQUIDO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_CREDITO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_AJUSTE else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_BRUTO_BC else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_PAGAMENTO_BRUTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_ABERTO_LIQ else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_JUROS else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_MULTA_CANCELAMENTO else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_PARC_APARELHO_LIQ else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(avg(case when U1M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U1M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U3M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U3M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U6M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U6M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U9M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U9M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(avg(case when U12M = 1 and IND_FRAUDE = 'S' then VAL_FAT_LIQ_JM_MC else NULL end),2) as VL_MED_U12M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO
    
FROM df_temp_01
GROUP BY NUM_CPF
ORDER BY NUM_CPF
""")

df_temp_02.createOrReplaceTempView("df_temp_02")
df_temp_02.count()


In [0]:
print('lista de colunas para tipar')

for col_name in spark.table("df_temp_02").columns:
    if col_name.startswith("VL_MED"):
        print(f"{col_name}")

## Criando variáveis explicativas de segunda camada

In [0]:
df_temp_03 = spark.sql("""

SELECT
    *,
    -- Razões para FAT_LIQUIDO_ATRASO
    round(try_divide(VL_MED_U1M_FAT_LIQUIDO_ATRASO , VL_MED_U3M_FAT_LIQUIDO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_LIQUIDO_ATRASO , VL_MED_U6M_FAT_LIQUIDO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_LIQUIDO_ATRASO , VL_MED_U9M_FAT_LIQUIDO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_LIQUIDO_ATRASO , VL_MED_U12M_FAT_LIQUIDO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_LIQ_ATRASO,
    
    -- Razões para FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_FAT_BRUTO_ATRASO , VL_MED_U3M_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_BRUTO_ATRASO , VL_MED_U6M_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_BRUTO_ATRASO , VL_MED_U9M_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_BRUTO_ATRASO , VL_MED_U12M_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_BRUTO_ATRASO,
    
    -- Razões para FAT_CREDITO_ATRASO
    round(try_divide(VL_MED_U1M_FAT_CREDITO_ATRASO , VL_MED_U3M_FAT_CREDITO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_CREDITO_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_CREDITO_ATRASO , VL_MED_U6M_FAT_CREDITO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_CREDITO_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_CREDITO_ATRASO , VL_MED_U9M_FAT_CREDITO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_CREDITO_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_CREDITO_ATRASO , VL_MED_U12M_FAT_CREDITO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_CREDITO_ATRASO,
    
    -- Razões para FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_FAT_AJUSTE_ATRASO , VL_MED_U3M_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_AJUSTE_ATRASO , VL_MED_U6M_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_AJUSTE_ATRASO , VL_MED_U9M_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_AJUSTE_ATRASO , VL_MED_U12M_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_AJUSTE_ATRASO,
    
    -- Razões para FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para FAT_PAGAMENTO_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_FAT_PAGAMENTO_BRUTO_ATRASO , VL_MED_U3M_FAT_PAGAMENTO_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_PAGAMENTO_BRUTO_ATRASO , VL_MED_U6M_FAT_PAGAMENTO_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_PAGAMENTO_BRUTO_ATRASO , VL_MED_U9M_FAT_PAGAMENTO_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_PAGAMENTO_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_PAGAMENTO_BRUTO_ATRASO , VL_MED_U12M_FAT_PAGAMENTO_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_PAGAMENTO_BRUTO_ATRASO,
    
    -- Razões para FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_FAT_ABERTO_ATRASO , VL_MED_U3M_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_ABERTO_ATRASO , VL_MED_U6M_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_ABERTO_ATRASO , VL_MED_U9M_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_ABERTO_ATRASO , VL_MED_U12M_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_ABERTO_ATRASO,
    
    -- Razões para FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_MULTA_JUROS_ATRASO , VL_MED_U3M_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_MULTA_JUROS_ATRASO , VL_MED_U6M_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_MULTA_JUROS_ATRASO , VL_MED_U9M_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_MULTA_JUROS_ATRASO , VL_MED_U12M_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_MULTA_JUROS_ATRASO,
    
    -- Razões para MULTA_CANCELAMENTO_ATRASO
    round(try_divide(VL_MED_U1M_MULTA_CANCELAMENTO_ATRASO , VL_MED_U3M_MULTA_CANCELAMENTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_MULTA_CANCELAMENTO_ATRASO,
    round(try_divide(VL_MED_U3M_MULTA_CANCELAMENTO_ATRASO , VL_MED_U6M_MULTA_CANCELAMENTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_MULTA_CANCELAMENTO_ATRASO,
    round(try_divide(VL_MED_U6M_MULTA_CANCELAMENTO_ATRASO , VL_MED_U9M_MULTA_CANCELAMENTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_MULTA_CANCELAMENTO_ATRASO,
    round(try_divide(VL_MED_U9M_MULTA_CANCELAMENTO_ATRASO , VL_MED_U12M_MULTA_CANCELAMENTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_MULTA_CANCELAMENTO_ATRASO,
    
    -- Razões para PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FAT_LIQ_JM_MC_ATRASO,
    
    -- Razões para FX_AGING_FAT_ATRASO
    round(try_divide(VL_MED_U1M_FX_AGING_FAT_ATRASO , VL_MED_U3M_FX_AGING_FAT_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FX_AGING_FAT_ATRASO,
    round(try_divide(VL_MED_U3M_FX_AGING_FAT_ATRASO , VL_MED_U6M_FX_AGING_FAT_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FX_AGING_FAT_ATRASO,
    round(try_divide(VL_MED_U6M_FX_AGING_FAT_ATRASO , VL_MED_U9M_FX_AGING_FAT_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FX_AGING_FAT_ATRASO,
    round(try_divide(VL_MED_U9M_FX_AGING_FAT_ATRASO , VL_MED_U12M_FX_AGING_FAT_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FX_AGING_FAT_ATRASO,
    
    -- Razões para FX_AGING_DIVIDA_ATRASO
    round(try_divide(VL_MED_U1M_FX_AGING_DIVIDA_ATRASO , VL_MED_U3M_FX_AGING_DIVIDA_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FX_AGING_DIVIDA_ATRASO,
    round(try_divide(VL_MED_U3M_FX_AGING_DIVIDA_ATRASO , VL_MED_U6M_FX_AGING_DIVIDA_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FX_AGING_DIVIDA_ATRASO,
    round(try_divide(VL_MED_U6M_FX_AGING_DIVIDA_ATRASO , VL_MED_U9M_FX_AGING_DIVIDA_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FX_AGING_DIVIDA_ATRASO,
    round(try_divide(VL_MED_U9M_FX_AGING_DIVIDA_ATRASO , VL_MED_U12M_FX_AGING_DIVIDA_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FX_AGING_DIVIDA_ATRASO,
    
    -- Razões para FX_TEMPO_BASE_ATRASO
    round(try_divide(VL_MED_U1M_FX_TEMPO_BASE_ATRASO , VL_MED_U3M_FX_TEMPO_BASE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FX_TEMPO_BASE_ATRASO,
    round(try_divide(VL_MED_U3M_FX_TEMPO_BASE_ATRASO , VL_MED_U6M_FX_TEMPO_BASE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FX_TEMPO_BASE_ATRASO,
    round(try_divide(VL_MED_U6M_FX_TEMPO_BASE_ATRASO , VL_MED_U9M_FX_TEMPO_BASE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FX_TEMPO_BASE_ATRASO,
    round(try_divide(VL_MED_U9M_FX_TEMPO_BASE_ATRASO , VL_MED_U12M_FX_TEMPO_BASE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FX_TEMPO_BASE_ATRASO,
    
    -- Razões para FX_AGING_PROX_FECH_ATRASO
    round(try_divide(VL_MED_U1M_FX_AGING_PROX_FECH_ATRASO , VL_MED_U3M_FX_AGING_PROX_FECH_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_FX_AGING_PROX_FECH_ATRASO,
    round(try_divide(VL_MED_U3M_FX_AGING_PROX_FECH_ATRASO , VL_MED_U6M_FX_AGING_PROX_FECH_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_FX_AGING_PROX_FECH_ATRASO,
    round(try_divide(VL_MED_U6M_FX_AGING_PROX_FECH_ATRASO , VL_MED_U9M_FX_AGING_PROX_FECH_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_FX_AGING_PROX_FECH_ATRASO,
    round(try_divide(VL_MED_U9M_FX_AGING_PROX_FECH_ATRASO , VL_MED_U12M_FX_AGING_PROX_FECH_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_FX_AGING_PROX_FECH_ATRAS,

    -- Razões para IND_PDD_S_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_LIQ_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_LIQ_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_LIQ_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_LIQ_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_LIQ_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_CRED_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_CRED_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_CRED_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_CRED_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_CRED_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_PDD_S_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_PDD_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_PDD_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_PDD_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_PDD_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_PDD_S_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_MULTA_CANC_ATRASO , VL_MED_U3M_IND_PDD_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_MULTA_CANC_ATRASO , VL_MED_U6M_IND_PDD_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_MULTA_CANC_ATRASO , VL_MED_U9M_IND_PDD_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_MULTA_CANC_ATRASO , VL_MED_U12M_IND_PDD_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_MULTA_CANC_ATRASO,
    
    -- Razões para IND_PDD_S_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_PDD_S_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_S_FAT_LIQ_JM_MC_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_LIQ_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_LIQ_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_LIQ_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_LIQ_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_LIQ_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_CRED_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_CRED_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_CRED_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_CRED_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_CRED_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_PDD_N_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_PDD_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_PDD_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_PDD_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_PDD_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_PDD_N_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_MULTA_CANC_ATRASO , VL_MED_U3M_IND_PDD_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_MULTA_CANC_ATRASO , VL_MED_U6M_IND_PDD_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_MULTA_CANC_ATRASO , VL_MED_U9M_IND_PDD_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_MULTA_CANC_ATRASO , VL_MED_U12M_IND_PDD_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_MULTA_CANC_ATRASO,
     
    -- Razões para IND_PDD_N_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_PDD_N_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PDD_N_FAT_LIQ_JM_MC_ATRASO,

    
    -- Razões para IND_WO_R_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_LIQ_ATRASO , VL_MED_U3M_IND_WO_R_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_LIQ_ATRASO , VL_MED_U6M_IND_WO_R_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_LIQ_ATRASO , VL_MED_U9M_IND_WO_R_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_LIQ_ATRASO , VL_MED_U12M_IND_WO_R_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_LIQ_ATRASO,
    
    -- Razões para IND_WO_R_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_WO_R_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_WO_R_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_WO_R_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_WO_R_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_WO_R_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_CRED_ATRASO , VL_MED_U3M_IND_WO_R_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_CRED_ATRASO , VL_MED_U6M_IND_WO_R_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_CRED_ATRASO , VL_MED_U9M_IND_WO_R_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_CRED_ATRASO , VL_MED_U12M_IND_WO_R_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_CRED_ATRASO,
    
    -- Razões para IND_WO_R_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_WO_R_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_WO_R_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_WO_R_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_WO_R_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_WO_R_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_WO_R_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_WO_R_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_WO_R_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_WO_R_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_WO_R_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_WO_R_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_WO_R_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_WO_R_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_WO_R_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_WO_R_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_WO_R_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_WO_R_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_WO_R_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_WO_R_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_WO_R_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_WO_R_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_WO_R_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_MULTA_CANC_ATRASO , VL_MED_U3M_IND_WO_R_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_MULTA_CANC_ATRASO , VL_MED_U6M_IND_WO_R_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_MULTA_CANC_ATRASO , VL_MED_U9M_IND_WO_R_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_MULTA_CANC_ATRASO , VL_MED_U12M_IND_WO_R_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_MULTA_CANC_ATRASO,
    
    -- Razões para IND_WO_R_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_WO_R_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_R_FAT_LIQ_JM_MC_ATRASO,

    -- Razões para IND_WO_W_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_LIQ_ATRASO , VL_MED_U3M_IND_WO_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_LIQ_ATRASO , VL_MED_U6M_IND_WO_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_LIQ_ATRASO , VL_MED_U9M_IND_WO_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_LIQ_ATRASO , VL_MED_U12M_IND_WO_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_LIQ_ATRASO,
    
    -- Razões para IND_WO_W_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_WO_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_WO_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_WO_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_WO_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_WO_W_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_CRED_ATRASO , VL_MED_U3M_IND_WO_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_CRED_ATRASO , VL_MED_U6M_IND_WO_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_CRED_ATRASO , VL_MED_U9M_IND_WO_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_CRED_ATRASO , VL_MED_U12M_IND_WO_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_CRED_ATRASO,
    
    -- Razões para IND_WO_W_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_WO_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_WO_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_WO_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_WO_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_WO_W_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_WO_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_WO_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_WO_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_WO_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_WO_W_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_WO_W_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_WO_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_WO_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_WO_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_WO_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_WO_W_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_WO_W_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_WO_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_WO_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_WO_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_WO_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_WO_W_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_MULTA_CANC_ATRASO , VL_MED_U3M_IND_WO_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_MULTA_CANC_ATRASO , VL_MED_U6M_IND_WO_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_MULTA_CANC_ATRASO , VL_MED_U9M_IND_WO_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_MULTA_CANC_ATRASO , VL_MED_U12M_IND_WO_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_MULTA_CANC_ATRASO,
    
    -- Razões para IND_WO_W_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_WO_W_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_WO_W_FAT_LIQ_JM_MC_ATRASO,
   
    -- Razões para IND_PCCR_W_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_LIQ_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_LIQ_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_LIQ_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_LIQ_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_LIQ_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_CRED_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_CRED_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_CRED_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_CRED_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_CRED_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_PCCR_W_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_PCCR_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_PCCR_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_PCCR_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_PCCR_W_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_PCCR_W_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_MULTA_CANC_ATRASO , VL_MED_U3M_IND_PCCR_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_MULTA_CANC_ATRASO , VL_MED_U6M_IND_PCCR_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_MULTA_CANC_ATRASO , VL_MED_U9M_IND_PCCR_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_MULTA_CANC_ATRASO , VL_MED_U12M_IND_PCCR_W_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_MULTA_CANC_ATRASO,
    
    -- Razões para IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_W_FAT_LIQ_JM_MC_ATRASO,
  
    -- Razões para IND_PCCR_C_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_LIQ_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_LIQ_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_LIQ_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_LIQ_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_LIQ_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_CRED_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_CRED_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_CRED_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_CRED_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_CRED_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_AJUSTE_ATRAS,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_AJUSTE_ATRASO,

    -- Razões para IND_PCCR_C_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_PCCR_C_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_PCCR_C_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_PCCR_C_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_PCCR_C_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_PCCR_C_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_PCCR_C_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_MULTA_CANC_ATRASO , VL_MED_U3M_IND_PCCR_C_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_MULTA_CANC_ATRASO , VL_MED_U6M_IND_PCCR_C_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_MULTA_CANC_ATRASO , VL_MED_U9M_IND_PCCR_C_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_MULTA_CANC_ATRASO , VL_MED_U12M_IND_PCCR_C_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_MULTA_CANC_ATRASO,
    
    -- Razões para IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PCCR_C_FAT_LIQ_JM_MC_ATRASO,
  
    -- Razões para IND_ACA_N_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_LIQ_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_LIQ_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_LIQ_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_LIQ_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_LIQ_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_CRED_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_CRED_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_CRED_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_CRED_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_CRED_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_ACA_N_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_ACA_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_ACA_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_ACA_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_ACA_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_ACA_N_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_MULTA_CANC_ATRASO , VL_MED_U3M_IND_ACA_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_MULTA_CANC_ATRASO , VL_MED_U6M_IND_ACA_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_MULTA_CANC_ATRASO , VL_MED_U9M_IND_ACA_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_MULTA_CANC_ATRASO , VL_MED_U12M_IND_ACA_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_MULTA_CANC_ATRASO,
    
    -- Razões para IND_ACA_N_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_ACA_N_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_N_FAT_LIQ_JM_MC_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_LIQ_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_LIQ_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_LIQ_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_LIQ_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_LIQ_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_CRED_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_CRED_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_CRED_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_CRED_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_CRED_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_ACA_S_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_ACA_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_ACA_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_ACA_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_ACA_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_ACA_S_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_MULTA_CANC_ATRASO , VL_MED_U3M_IND_ACA_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_MULTA_CANC_ATRASO , VL_MED_U6M_IND_ACA_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_MULTA_CANC_ATRASO , VL_MED_U9M_IND_ACA_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_MULTA_CANC_ATRASO , VL_MED_U12M_IND_ACA_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_MULTA_CANC_ATRASO,
    
    -- Razões para IND_ACA_S_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_ACA_S_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_ACA_S_FAT_LIQ_JM_MC_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_LIQ_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_CRED_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_CRED_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_CRED_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_CRED_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_CRED_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_MULTA_CANC_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_S_FAT_LIQ_JM_MC_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_LIQ_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_CRED_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_CRED_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_CRED_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_CRED_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_CRED_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_MULTA_CANC_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_PRIM_FAT_N_FAT_LIQ_JM_MC_ATRASO,
  
    -- Razões para IND_FRAUDE_N_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_LIQ_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_LIQ_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_LIQ_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_LIQ_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_LIQ_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_CRED_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_CRED_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_CRED_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_CRED_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_CRED_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_FRAUDE_N_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_FRAUDE_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_FRAUDE_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_FRAUDE_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_FRAUDE_N_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_FRAUDE_N_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_MULTA_CANC_ATRASO , VL_MED_U3M_IND_FRAUDE_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_MULTA_CANC_ATRASO , VL_MED_U6M_IND_FRAUDE_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_MULTA_CANC_ATRASO , VL_MED_U9M_IND_FRAUDE_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_MULTA_CANC_ATRASO , VL_MED_U12M_IND_FRAUDE_N_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_MULTA_CANC_ATRASO,
    
    -- Razões para IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_N_FAT_LIQ_JM_MC_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_LIQ_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_LIQ_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_LIQ_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_LIQ_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_LIQ_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_BRUTO_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_BRUTO_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_BRUTO_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_BRUTO_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_BRUTO_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_CRED_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_CRED_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_CRED_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_CRED_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_CRED_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_CRED_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_CRED_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_AJUSTE_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_AJUSTE_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_BRUTO_BC_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_PAGT_BRUTO_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_ABERTO_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_ABERTO_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_ABERTO_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_ABERTO_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_ABERTO_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_ABERTO_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_ABERTO_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_ABERTO_LIQ_ATRASO,
    
    -- Razões para IND_FRAUDE_S_MULTA_JUROS_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_MULTA_JUROS_ATRASO , VL_MED_U3M_IND_FRAUDE_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_MULTA_JUROS_ATRASO , VL_MED_U6M_IND_FRAUDE_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_MULTA_JUROS_ATRASO , VL_MED_U9M_IND_FRAUDE_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_MULTA_JUROS_ATRASO , VL_MED_U12M_IND_FRAUDE_S_MULTA_JUROS_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_MULTA_JUROS_ATRASO,
    
    -- Razões para IND_FRAUDE_S_MULTA_CANC_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_MULTA_CANC_ATRASO , VL_MED_U3M_IND_FRAUDE_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_MULTA_CANC_ATRASO , VL_MED_U6M_IND_FRAUDE_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_MULTA_CANC_ATRASO , VL_MED_U9M_IND_FRAUDE_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_MULTA_CANC_ATRASO , VL_MED_U12M_IND_FRAUDE_S_MULTA_CANC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_MULTA_CANC_ATRASO,
    
    -- Razões para IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U3M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U6M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U9M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO , VL_MED_U12M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_PARC_APARELHO_LIQ_ATRASO,
    
    -- Razões para IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO
    round(try_divide(VL_MED_U1M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U3M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U1M_U3M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U3M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U6M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U3M_U6M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U6M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U9M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U6M_U9M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO,
    round(try_divide(VL_MED_U9M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO , VL_MED_U12M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRASO), 2) as VL_RAZ_MED_U9M_U12M_IND_FRAUDE_S_FAT_LIQ_JM_MC_ATRAS

    
FROM df_temp_02

""")

df_temp_03.createOrReplaceTempView("df_temp_03")
df_temp_03.count()


In [0]:
from pyspark.sql.functions import lit

df_temp_04 = df_book_atraso_01.alias("t1") \
    .join(df_temp_03.alias("t2"), "NUM_CPF", "left") \
    .withColumn("SAFRA", lit(data_exec_inicial)) \
    .withColumn("DATPROC", lit(dthproc))

df_temp_04.createOrReplaceTempView("df_temp_04")
df_temp_04.count()

In [0]:
display(df_temp_04.limit(10))

In [0]:
qtd_variaveis_explicativas = len(df_temp_04.columns) - 3
print(qtd_variaveis_explicativas)

In [0]:

from delta.tables import DeltaTable

silver_table = "hackathon2025.silver.book_atraso"

if not spark.catalog.tableExists(silver_table):
    print("Tabela silver não existe. Criando...")

    (
        df_temp_04
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table) 
    )

    print("Tabela silver criada com sucesso")

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forName(spark, silver_table)

    (
        delta_silver.alias("t")
        .merge(
            df_temp_04.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Dados inseridos com sucesso...")



In [0]:
name = "book_atraso"

df_controle = spark.sql("""
    SELECT
        '{name_table}'        AS nome_tabela,
        SAFRA                 AS safra,
        COUNT(*)              AS qtd_registros,
        current_timestamp()   AS datproc
    FROM df_temp_04
    GROUP BY SAFRA
    ORDER BY SAFRA
""".format(name_table=name))

display(df_controle)

In [0]:
silver_table_controle = "hackathon2025.silver.controle"
if not spark.catalog.tableExists(silver_table_controle):
    print("Tabela silver_controle não existe. Criando...")

    (
        df_controle
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table_controle)  
    )

    print("Tabela silver_controle criada com sucesso")

else:
    print("Tabela de controle existe. Inserindo novo registro...")

    delta_silver = DeltaTable.forName(spark, silver_table_controle)

    (
        df_controle
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table_controle)
    )
    print("Dados inseridos com sucesso...")